# FER2013: Facial Expression Recognition
მთავარი Colab notebook. აქ ხდება გარემოს მომზადება, sanity check-ები და ექსპერიმენტები wandb-ზე ლოგირებით.

Runtime > Change runtime type > GPU აუცილებლად ჩართე.

## 1: რეპო და ბიბლიოთეკები

In [ ]:
!git clone https://github.com/GigiSichinava/ML-Assignment-4.git
%cd ML-Assignment-4
!pip install -q wandb kaggle

## 2: credentials (Colab Secrets-დან)
მარცხნივ 🔑 იკონაში ჩაამატე 3 secret: KAGGLE_USERNAME, KAGGLE_KEY, WANDB_API_KEY.
თითოეულს ჩართე 'Notebook access'. ეს ერთხელ კეთდება, მერე ავტომატურად მუშაობს.

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
print('creds loaded')

## 3: მონაცემები
ჯერ Kaggle-ზე competition-ის გვერდზე 'Join Competition'-ს დააჭირე (წესების მიღება),
თორემ download 403-ს დააბრუნებს.

In [ ]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-*.zip -d data/
from src.data import prepare_data
prepare_data('data')

## 4: wandb
WANDB_API_KEY უკვე env-შია, ანუ login-ი მყისიერია, არაფერს გეკითხება.

In [ ]:
import wandb
wandb.login()

## 5: sanity check-ები (forward და backward)
სრულ ტრენინგამდე ვამოწმებ, რომ მოდელი და loop გამართულია.

In [ ]:
import torch
from src.config import Config
from src.data import get_dataloaders
from src.models import get_model
from src.utils import set_seed, count_params, check_initial_loss, overfit_small_batch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(42)

cfg = Config(arch='SmallCNN')
train_loader, val_loader, test_loader, class_weights = get_dataloaders(cfg)
model = get_model(cfg).to(device)
print('params:', count_params(model))

# forward: საწყისი loss ~ ln(7) = 1.946
check_initial_loss(model, train_loader, device)
# backward: პატარა batch-ის overfit, ~100% train acc
overfit_small_batch(get_model(cfg).to(device), train_loader, device, n=20, steps=200)

## 6: ექსპერიმენტები არქიტექტურების მიხედვით
თითო preset ცალკე wandb run-ია. პატარადან ვიწყებ და ვამატებ სირთულეს.
თითო ნაბიჯის შემდეგ wandb-ის გრაფიკს ვუყურებ და readme-ში ვწერ, რატომ გადავდგი შემდეგი ნაბიჯი.

In [ ]:
from src.config import PRESETS
from src.train import train_model
from src.utils import plot_history, plot_confusion

# 1: underfit baseline
cfg = PRESETS['mlp_baseline']
model, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title=cfg.run_name);

In [ ]:
# 2: SmallCNN
cfg = PRESETS['smallcnn']
model, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title=cfg.run_name)
plot_confusion(y_true, y_pred);

In [ ]:
# 3: DeeperCNN regularization-ის გარეშე, overfit
cfg = PRESETS['deepcnn_overfit']
model, history, best, _ = train_model(cfg, device=device)
plot_history(history, title=cfg.run_name);

In [ ]:
# 4: DeeperCNN რეგულარიზებული, overfit-ის გასწორება
cfg = PRESETS['deepcnn_regularized']
best_model, history, best, (y_true, y_pred) = train_model(cfg, device=device)
plot_history(history, title=cfg.run_name)
plot_confusion(y_true, y_pred);

## 7: ჰიპერპარამეტრების გადარჩევა
თითო არქიტექტურაში რამდენიმე run სხვადასხვა ჰიპერპარამეტრით.
ყველა ერთ group-ში ჯდება და wandb-ში ერთმანეთს ვადარებ. მაგალითად lr-ის გავლენა:

In [ ]:
# SmallCNN, lr-ის sweep (3 run ერთ group-ში)
for lr in [1e-2, 1e-3, 1e-4]:
    cfg = Config(arch='SmallCNN', run_name=f'smallcnn_lr{lr}',
                 lr=lr, dropout=0.25, epochs=25)
    train_model(cfg, device=device)

In [ ]:
# DeeperCNN, dropout-ის გავლენა overfit-ზე
for d in [0.0, 0.3, 0.5]:
    cfg = Config(arch='DeeperCNN', run_name=f'deepcnn_drop{d}',
                 dropout=d, weight_decay=1e-4, augment=True,
                 scheduler='cosine', epochs=40)
    train_model(cfg, device=device)

## 8: Kaggle submission (საუკეთესო მოდელით)

In [ ]:
from src.train import make_submission
make_submission(best_model, PRESETS['deepcnn_regularized'], out_path='submission.csv', device=device)
# !kaggle competitions submit -c challenges-in-representation-learning-facial-expression-recognition-challenge -f submission.csv -m 'deepcnn_reg'